# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DawoodAhmadAnalyst/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My lane
I chose Lane 2 – Refresh / Content Opportunity Scoring.
The goal of this project is to help content editors prioritize which pages should be reviewed first for refresh, expansion, monitoring, or other content improvements. Rather than trying to automate SEO decisions, the project provides a ranked list of pages that deserve human attention based on observable search and engagement signals.
I selected this lane because the starter dataset already contains useful signals such as impressions, CTR, engagement rate, average position, content age, freshness, and traffic trends. These signals make it possible to build a transparent ranking system that supports editorial decision-making. The starter pipeline validates with a client-holdout split (32 unique clients in this dataset), which matters for this lane specifically — a ranking that only works for clients it has already seen isn't useful for prioritizing a new client's backlog.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question
### Research question
Which content pages should be reviewed first so editors spend their limited review time on the highest-value refresh opportunities?
### Decision
The decision is which content pages should be prioritized for manual review.
### Unit of analysis
One row represents one content page.
### Output
A ranked list of pages with a priority score and reason codes indicating why each page deserves review.
### Who acts?
A content editor or SEO specialist, working through a limited review queue (e.g. the top 50 pages per cycle).
### Action
The reviewer examines the highest-ranked pages and decides whether they should be refreshed, expanded, monitored, or otherwise improved.
### Cost of a wrong recommendation
A false positive sends a reviewer to a page that was already performing fine — wasted review time. A false negative means a genuinely declining page never reaches the queue — lost traffic that goes unaddressed.
This isn't abstract: the starter pipeline's own results (Notebook 01) show the gap concretely. At Precision@50, the hand-written baseline rule gets ~12 of its top 50 picks right (0.240); the random forest gets ~37 right (0.740). That means the baseline sends reviewers to roughly 38 pages per 50-page batch that weren't actually declining, versus about 13 for the learned ranking — nearly 3x the wasted review effort under the same capacity constraint.
### Why ML helps
A single threshold rule (e.g. "stale AND visible") can't capture how these signals interact. For example, the depth-2 decision tree in Notebook 02 didn't just threshold on one signal — it split first on impressions_90d, then on content_age_days within that branch, effectively learning "high-impression pages are risky specifically when they're not too old." That's a two-signal interaction a flat if/else rule doesn't express cleanly. A learned model — even a shallow, readable one — can pick up on interactions like this that a fixed rule misses, without needing to be complex or opaque. The goal is still to support human decision-making, not replace it.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique clients: {df['client_id'].nunique()}")

declining = (df["trend_direction"] == "down").sum()

print(f"Pages labelled 'down': {declining:,}")
print(f"Percentage declining: {declining / len(df):.1%}")

Rows: 30,000
Columns: 44
Unique clients: 32
Pages labelled 'down': 16,262
Percentage declining: 54.2%


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## Careful words
This project is designed as a decision-support system rather than an automated decision maker.
A high priority score does not guarantee that refreshing a page will improve its performance. Instead, the score identifies pages that appear to deserve manual review based on observable signals.
The starter dataset's label — trend_direction == "down" — is a proxy computed from the current window (trend_pct), not a future observed outcome. As Notebook 02 shows directly, feeding trend_pct into a model as a feature reproduces the label almost perfectly (Precision@50 = 1.000) precisely because the label is derived from it — that's leakage, not a discovery. For this reason, the starter label is treated as a beginner scaffold, not the intended capstone target.
For the capstone, this proxy will be replaced with a genuine future-window label — e.g. features from the prior 90 days predicting decline or recovery over the next 30 days — built from the warehouse daily facts (fact_content_daily_performance), with a leakage audit run before any modeling begins.
Given this, the project's claims stay bounded: observational and directional ("pages with these characteristics were more likely to be flagged"), never causal ("refreshing this page will increase its traffic") and never about search engine ranking algorithms themselves.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Reviewable population — the rows the pipeline actually scores
# (per docs/ml-intern-dataset-and-lane-guide.md: impressions_90d > 0 and content_age_days >= 90)
reviewable = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]

print(f"Total rows: {len(df):,}")
print(f"Reviewable rows (impressions_90d > 0 AND content_age_days >= 90): {len(reviewable):,}")
print(f"Share of dataset that's actually in-scope for review: {len(reviewable)/len(df):.1%}")

declining_reviewable = (reviewable["trend_direction"] == "down").sum()
print(f"Declining rate within reviewable population: {declining_reviewable/len(reviewable):.1%}")

Total rows: 30,000
Reviewable rows (impressions_90d > 0 AND content_age_days >= 90): 30,000
Share of dataset that's actually in-scope for review: 100.0%
Declining rate within reviewable population: 54.2%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.